In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# the hindenburg omen is about market internals, not just
# the index price. I need individual stock data to calculate
# how many stocks are hitting new highs vs new lows.
#
# I'm using ~85 major stocks as a proxy for NYSE breadth.
# the actual omen uses all NYSE traded issues (~3000+).
# my proxy is what one trash panda can download without
# my laptop catching fire, which feels thematically appropriate.

TICKERS = [
    # Technology
    'AAPL', 'MSFT', 'INTC', 'CSCO', 'ORCL', 'IBM', 'TXN', 'QCOM',
    'ADBE', 'HPQ', 'AMD', 'MU',
    # Healthcare
    'JNJ', 'PFE', 'MRK', 'ABT', 'BMY', 'AMGN', 'GILD', 'MDT',
    'UNH', 'LLY',
    # Financials
    'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'AXP', 'BK', 'USB',
    'PNC', 'MET',
    # Consumer Discretionary
    'HD', 'MCD', 'NKE', 'SBUX', 'TGT', 'LOW', 'TJX', 'F',
    # Consumer Staples
    'PG', 'KO', 'PEP', 'WMT', 'CL', 'GIS', 'MO', 'SYY',
    # Industrials
    'GE', 'MMM', 'CAT', 'BA', 'HON', 'UPS', 'LMT', 'DE', 'EMR',
    # Energy
    'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'OXY', 'VLO', 'HAL',
    # Utilities
    'DUK', 'SO', 'NEE', 'D', 'AEP', 'SRE', 'EXC', 'ED',
    # Telecom / Media
    'T', 'VZ', 'CMCSA', 'DIS',
    # Materials / REITs
    'APD', 'ECL', 'FCX', 'NEM', 'NUE', 'SPG', 'PLD',
]

print(f"Downloading data for {len(TICKERS)} stocks...")
raw = yf.download(TICKERS, start='2000-01-01', end='2025-01-01')
closes = raw['Close']

# also get S&P 500 index
sp500 = yf.download('^GSPC', start='2000-01-01', end='2025-01-01')
sp500_close = sp500['Close'].squeeze()
sma_50 = sp500_close.rolling(50).mean()

print(f"Got data for {closes.shape[1]} stocks")

# breadth metrics
# new 52-week high: today's close equals the 252-day rolling max
# new 52-week low: today's close equals the 252-day rolling min
rolling_max = closes.rolling(252, min_periods=252).max()
rolling_min = closes.rolling(252, min_periods=252).min()

new_highs = (closes >= rolling_max) & closes.notna()
new_lows = (closes <= rolling_min) & closes.notna()
stocks_available = closes.notna().sum(axis=1)

pct_new_highs = new_highs.sum(axis=1) / stocks_available
pct_new_lows = new_lows.sum(axis=1) / stocks_available

# McClellan Oscillator from advance/decline data
daily_returns = closes.pct_change()
advances = (daily_returns > 0).sum(axis=1)
declines = (daily_returns < 0).sum(axis=1)
ad_diff = advances - declines
mcclellan = ad_diff.ewm(span=19).mean() - ad_diff.ewm(span=39).mean()

# assemble the indicator
breadth = pd.DataFrame({
    'pct_new_highs': pct_new_highs,
    'pct_new_lows': pct_new_lows,
    'mcclellan': mcclellan,
    'n_stocks': stocks_available,
    'sp500': sp500_close,
    'sma_50': sma_50,
}).dropna()

# Hindenburg Omen: all four conditions simultaneously
THRESHOLD = 0.022
breadth['omen'] = (
    (breadth['pct_new_highs'] > THRESHOLD) &
    (breadth['pct_new_lows'] > THRESHOLD) &
    (breadth['sp500'] > breadth['sma_50']) &
    (breadth['mcclellan'] < 0) &
    (breadth['pct_new_highs'] < 2 * breadth['pct_new_lows'] + 0.001)
).astype(int)

# cluster signals within 30 days into one event.
# proponents say the omen is "confirmed" when multiple
# fire in a short window. I'm being generous.
omen_dates = breadth[breadth['omen'] == 1].index
clusters = []
cluster_start = None
for d in omen_dates:
    if cluster_start is None or (d - cluster_start).days > 30:
        cluster_start = d
        clusters.append(d)

print(f"\nHindenburg Omen Detection")
print(f"Period: 2000-2025")
print(f"Stocks in universe: {closes.shape[1]}")
print(f"Individual omen days: {len(omen_dates)}")
print(f"Signal clusters (30-day grouping): {len(clusters)}")
print(f"Average omens per year: {len(omen_dates) / 25:.1f}")

[*********************100%***********************]  85 of 85 completed
[*********************100%***********************]  1 of 1 completed

Got data for 85 stocks

Hindenburg Omen Detection
Period: 2000-2025
Stocks in universe: 85
Individual omen days: 72
Signal clusters (30-day grouping): 35
Average omens per year: 2.9
